### "Tickers and Company names are arbitrary, re-useable things." And we can't use composite_FIGI or CIK as unique identifiers (see Introduction). So this is the only method that works! 
This notebook is designed to handle one of the most complex challenges in building a stock price database: properly accounting for ticker symbol changes (renamings) over time. 
Because the Polygon.io API stores data by ticker symbol, which can change when a company rebrands or is acquired, the goal is to create a unified, historical record that links all past symbols for a company to its current one, ensuring a continuous price history without gaps or data loss.

### 💡 Problem: Ticker-Centric Data (FB ticker data stays with FB ticker and doesn't get passed to META on ticker change. BITF/KEEL, etc.)
A key challenge with Polygon.io data (and others like Norgate) is that it's "ticker-centric". If a company changes its ticker (e.g., from `A` to `B`), the historical price data for `A` and the new data for `B` are stored separately. This notebook aims to solve this by creating a mapping that tells you, for any given date, which ticker symbol to use for a specific company. As the notebook's author notes, without this, a backtest might incorrectly assume a stock stopped trading when it simply changed its ticker.

### 🔀 Where the Data Comes From: `stockanalysis.com` (Cheap/free Solution = free trial or $10 one-off, and for ongoing updates the site gives last 12 months ticker changes for FREE under the public "Recent ticker changes" public endpoint with no login required!)
The notebook explicitly states that building this mapping directly from Polygon.io's own `Ticker Events` endpoint is problematic and doesn't work for delisted companies. The author attempted an alternative method of inferring changes but found it "very messy".

The chosen solution is to source ticker change data from [**stockanalysis.com**](https://stockanalysis.com/actions/changes/), which requires a subscription (noted as costing $10/month and having a free first month). The user is instructed to manually download these CSV files for each year and place them in the folder `stockanalysis/raw/ticker_changes/`.

### ⚙️ What This Notebook Does (A Multi-Step Process)
The notebook processes the data from `stockanalysis.com` and integrates it with the main ticker list to produce a definitive master record.

1.  **Aggregate Data**: It reads all the manually downloaded CSV files from the `ticker_changes` folder and combines them into a single DataFrame.
2.  **Process & Map Changes**: The core logic involves a loop that goes through the existing ticker list and, for each ticker, looks for a change record on its "end date." If a change is found, the notebook:
    *   Merges the stock's data record with its new ticker symbol.
    *   Updates the main ticker list (`tickers_v4.csv`) to reflect the current symbol.
    *   Logs the historical change in a new column called `tickers_old`, which stores a list of `(date_of_change, old_ticker)` pairs. This creates a full audit trail.
3.  **Create the Final Master List**: The final output is a new file, `tickers_v5.csv`. This file serves as the definitive ticker list for the entire database, ensuring that every company's price history can be traced correctly through all its symbol changes.

# 8.1 Merging ticker changes
*(For myself I skip this part. Renaming give to much headache, even though it's not that important for short-term signals. Also it does not affect the price itself.)*
This is optional. If you want it ticker-centric or don't want to get a stockanalysis.com subscription, you can just skip this part.

**^^^ Dev: But we have to do it cos we trade multi-week strategies too! And this affects the stocks we trade! FB/META, BITF/KEEL !** 

To get a list of ticker changes, We can loop through all tickers and query <code>Ticker Events</code> but this only works with non-delisted companies. And although you can infer it based on the ticker list by looking at whether the cik or figi has changed, that is very messy. Because a company can stay the same even if the ticker and cik/figi change. I actually did it, and it did found that it did not match the Polygon <code>Ticker Events</code>. Then I stumbled on [stockanalysis.com](https://stockanalysis.com/actions/changes/) where you can find all ticker changes for only 10 bucks a month. The first month is even free. You have to manually download them for each year and put them in the <code>stockanalysis/raw/ticker_changes/</code> folder.

After merging those we will save the result to <code>raw/renamings.csv</code> which will contain the columns <code>['from', 'to', 'now', 'date']</code>.

In [ ]:
from tickers import get_tickers, get_ticker_changes
from times import get_market_dates, get_market_calendar, last_trading_date_before
from datetime import datetime, date, time
import pandas as pd
import shutil
import os
DATA_PATH = "../data/polygon/"
END_DATE = last_trading_date_before(date(2199, 4, 19))

In [ ]:
# This can be done once and then updates can be done with manually appending the list of ticker changes.
###
# Aggregate the csv's
all_ticker_changes = []
for file in os.listdir(DATA_PATH + "../stockanalysis/raw/ticker_changes/"):
    ticker_changes_year = pd.read_csv(DATA_PATH + "../stockanalysis/raw/ticker_changes/" + file, \
        parse_dates=True, index_col=0, usecols=["Date", "Old", "New"])
    all_ticker_changes.append(ticker_changes_year)

ticker_changes = pd.concat(all_ticker_changes)
ticker_changes = pd.concat(all_ticker_changes)
ticker_changes.rename(columns={"Old": "from", 
                               "New": "to"}, inplace=True)
ticker_changes.index.names = ['date']
ticker_changes.sort_index(inplace=True)
ticker_changes.to_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")

In [3]:
ticker_changes = pd.read_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")
print(len(ticker_changes))
ticker_changes[ticker_changes['from'] == "FB"]

4491


,date,from,to
4013,2022-06-09,FB,META


# More problems

Seems there are special conditions, such as 'delinquent', which adds an extra letter at the end of the ticker. E.g. AAII went delinquent and then the ticker became AAIIE. However these are not real ticker changes so it is not contained in the stockanalysis database. However we can easily infer it from our own ticker list: if the dates are consecutive and an extra letter is added, we can infer the ticker change. We will save this to <code>raw/inferred_renamings.csv.</code>

### TO-DO

There are a whopping 4347 ticker changes from 2003 to now that we have to take care of. But at least this was very easy to get.

Now we will use them to merge our data. We have to be aware that it is possible for a ticker to used multiple times, so the <code>ticker_changes.csv</code> may contain multiple of the same tickers in the 'from' and 'to' column. 

After processing the ticker changes we will create a <code>tickers_v5.csv</code> which will be our definitive ticker list. This contains a column 'tickers_old', which will contain a list of (date_of_change, ticker) pairs. So if A changes to B on day 2, and B changes to C on day 5, tickers_old for D will contain [[2, A], [5, B]].

The process will be as follows:
* As long as we have ticker changes to process
    * Loop through <code>tickers_v4.csv</code>.
        * Get the next trading date after 'end_date_data'.
        * Search in <code>tickers_changes.csv</code> if there is a ticker change on this date.
        * If it does:
            * The stock data will be merged.
            * In <code>tickers_v4.csv</code> we will change "ticker" to the new ticker and add a list [date, ticker] to "tickers_old".
            * All other rows will be merged such as "start_date". For identifiers such as FIGI we will take the last available value. For the ID we will keep the original. If we do not do this, we might run into problems with identical IDs.
            * The row of the old ticker will be deleted
            * **We need to restart the loop!** If we don't the following can happen: Let's assume that a ticker was renamed from A -> B -> C -> D but that the order in which it appears in our ticker list is C, D, A, B. Using our loop, C gets merged with D. Then the loop checks D, which has no renamings. Then A gets merged with B. Then B gets merged with C, however that is incorrect! B should be merged with the new D, which contains C. Any double+ renamings have the risk of being in the 'wrong order'.
                * For this to work, the ticker list must be sorted on end_date.
            * Of course we must not forget that there can be adjustments on day 1 of the ticker change. There should be laws to prohibit this.

Note: if a ticker A goes OTC and then comes back and changes to B, then we will have two files: one of the A before OTC and the A+B after OTC named B. This is as intended.

In [ ]:
tickers_v4 = get_tickers(v=4)
# QUICK BUG FIX, NEED TO REWRITE CODE TO MAKE IT CHRONIGICAL
market_dates = get_market_dates()
ticker_changes = get_ticker_changes()

tickers_v4.insert(loc = 2, column = 'tickers_old', value = [{} for _ in range(len(tickers_v4))])

while True:
    tickers_v4 = tickers_v4.sort_values(by='end_data').reset_index(drop=True)

    # tickers_v4 gets smaller by 1 element every time we run this loop.
    for index_from, row_from in tickers_v4.copy().iterrows():
        # Get values
        type_from = row_from['type']
        if type_from == "INDEX":
            continue
        id_from = row_from['ID']
        ticker_from = row_from['ticker']
        start_date_from = row_from['start_date']
        end_date_from = row_from['end_date']
        start_data_from = row_from['start_data']
        end_data_from = row_from['end_data']

        if end_data_from == END_DATE:
            continue

        start_data_to = market_dates[market_dates.index(end_data_from) + 1]

        # Get ticker changes 
        change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
        if change.empty:
            continue
        elif len(change) > 1:
            raise Exception("Duplicate!")
        ticker_to = change['to'].values[0]

        # Set values of new ticker
        row_to = tickers_v4[(tickers_v4['start_data'] == start_data_to) & (tickers_v4['ticker'] == ticker_to)]
        if row_to.empty:
            continue
        index_to = row_to.index[0]
        id_to = row_to['ID'].values[0]
        tickers_v4.loc[index_to, "tickers_old"][start_data_to.isoformat()] = ticker_from
        tickers_v4.loc[index_to, "start_date"] = start_date_from
        tickers_v4.loc[index_to, "start_data"] = start_data_from

        # Do the actual merging
        from_ = pd.read_parquet(DATA_PATH + f"processed/m1/{id_from}.parquet")
        to = pd.read_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet")

        # When a ticker changes, the adjustments carry over the the old ticker.
            # Get market close minute to calculate the adjustment factor, and adjust 'to'.
            # (Adjustment factor is the same throughout the day, so market close is arbitrarely chosen)
        calendar = get_market_calendar('datetime')
        start_data_to_dt = calendar.loc[start_data_to, 'regular_close']
        start_data_to_dt_bar = to.loc[start_data_to_dt]
        adjustment_factor = start_data_to_dt_bar['close'] / start_data_to_dt_bar['close_original']
        from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor)
        from_ = round(from_, 4)

        # Because companies like to be annoying, a split/dividend can take place at the same time as a ticker change. We have to account for this. An example is TYDE -> OCTO with a 50:1 reverse split. However this is much easier than 5_process_raw_data.ipynb, because there is at most a single split. This is rare, but there are still a handful of cases.
        if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_to}.csv"):
            adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_to}.csv", parse_dates=True, index_col=0)
            adjustments.index = pd.to_datetime(adjustments.index).date
            adjustments = adjustments[(adjustments.index == start_data_to)]

            # SPLIT ADJUSTMENT
            split = adjustments[adjustments.type == 'SPLIT']
            if len(split) > 0:
                split_amount = split['amount'][0]

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(split_amount)

            # DIVIDEND ADJUSTMENT - REUN is the only case, not clear what happened there, likely a 'special dividend'
            dividend = adjustments[adjustments.type == 'DIV']
            if len(dividend) > 0:
                print(ticker_to)
                market_hours = get_market_calendar()
                market_hours = market_hours[['regular_close']]

                cum_div_date = end_data_from
                cum_div_time = market_hours.loc[cum_div_date][0]
                cum_div_datetime = datetime.combine(cum_div_date, cum_div_time)
                cum_div_datetime = (from_[from_.index <= cum_div_datetime].index).max()
                cum_div_close = from_.loc[cum_div_datetime, 'close']
                dividend_amount = dividend['amount'][0]
                    
                adjustment_factor = 1 - dividend_amount/cum_div_close

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor)
            
            # ROUNDING
            if len(split) > 0 or len(dividend) > 0:
                from_ = round(from_, 4)
                from_['turnover'] = from_['turnover'].astype(int)

        # # If on the old ticker, there are divs/splits on start_data_to (start of new ticker), then something is wrong.
        # if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_from}.csv"):
        #     adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_from}.csv", parse_dates=True, index_col=0)
        #     adjustments.index = pd.to_datetime(adjustments.index).date
        #     adjustments = adjustments[(adjustments.index == start_data_to)]
        #     assert len(adjustments) == 0

        # Remove the 'from' ticker, then paste the 'from' and 'to' ticker to m1_renamed for debugging purposes.
        _ = shutil.move(DATA_PATH + f'processed/m1/{id_from}.parquet', DATA_PATH + f'processed/m1_renamed/{id_from}.parquet')
        _ = shutil.copyfile(DATA_PATH + f'processed/m1/{id_to}.parquet', DATA_PATH + f'processed/m1_renamed/{id_to}.parquet')

        pd.concat([from_, to]).to_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet", engine="fastparquet", row_group_offsets=25000)

        tickers_v4.drop(index_from, inplace=True)
        tickers_v4.reset_index(inplace=True, drop=True)
        
        print(f"Ticker change {ticker_from} -> {ticker_to} on {start_data_to} has been processed")
        print(f"{index_from/len(tickers_v4)*100:.1f}% | Length of tickers_v4 is {len(tickers_v4)}")
        break

    # If we have reached the end of the loop, it means we have processed everything. Then we can stop.
    if index_from >= (len(tickers_v4)-1):
        break
    
tickers_v4 = tickers_v4.sort_values(by='ID').reset_index(drop=True)

tickers_v4.to_csv("../data/tickers_v5.csv")

In [7]:
tickers_v5 = get_tickers(v=5)
renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 2] # These were renamed
print(len(renamings))
tickers_v5[tickers_v5['ticker'] == 'META']

1007


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
9369,META-2022-06-09,META,{'2022-06-09': 'FB'},"Meta Platforms, Inc. Class A Common Stock",True,2012-05-18,2024-04-19,2012-05-18,2024-04-19,CS,1326801.0,BBG000MM2P62


Tickers that were renamed multiple times:

In [15]:
multiple_renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 23]
print(len(multiple_renamings))
multiple_renamings.head(2)

0


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi


Now we have 5 tickers lists. These are:
1. Basic ticker list with a lot of incorrect duplications.
2. Duplications merged and incorrect tickers removed.
3. ETFs added.
4. Data start/end dates added.
5. Renamings merged.
Only the last should be used in backtesting.

If Polygon just provided these from the start, it would have saved countless hours. But at least I learned some Python I guess. And at least Polygon does not ask thousands.

# 8.2 Updates
Rerun the file after setting END_DATE and updating the list of renamings.